<a href="https://colab.research.google.com/github/omsoni/llm-rag-work/blob/rag_evaluation_scope/RAG_Medical_Assistant_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Address Github Compatibility for nbformat

In [ ]:
import json

with open('RAG_Medical_Assistant_Evaluation.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

if 'widgets' in nb.get('metadata', {}):
    for widget_key in nb['metadata']['widgets']:
        if 'state' not in nb['metadata']['widgets'][widget_key]:
            nb['metadata']['widgets'][widget_key]['state'] = {}

with open('RAG_Medical_Assistant_Evaluation.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

## Summary

### Objective

Build and Experiment with Evlaution model for RAG based Medical Assistant

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --no-cache-dir --no-deps -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 44.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install --upgrade huggingface_hub==0.35.3 pandas==2.2.2 langchain-community==0.3.31 chromadb==1.1.1  sentence-transformers==5.1.1  -q

In [ ]:
pip install diskcache llama-cpp-python==0.2.28 --no-deps --no-cache-dir -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 275.9 MB/s eta 0:00:00


In [ ]:
#Libraries for processing dataframes,text
import json,os
import pandas as pd
import numpy as np

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [ ]:
import textwrap
import warnings
warnings.filterwarnings('ignore')

#### Function to pretty print a collection

In [ ]:
def pretty_print_doc_collection(relevant_document_chunks):
  for i, chunk in enumerate(relevant_document_chunks):
    print(f"── Chunk {i+1} ──────────────────────────────────────────")
    print(f"Page   : {chunk['metadata'].get('page', 'N/A')}")
    print(f"Score  : {chunk['score']:.4f}")
    print(f"Text   :")
    print(textwrap.fill(chunk['text'], width=80))
    print()

# Question Answering using LLM

Following Google Colab T4 friendly LLMs on huggingface were analyzed and compared:

| Model | Params | Domain | Strengths | Weaknesses | Colab T4 Friendly | Notes |
|------|------|------|------|------|------|------|
| meta-llama/Meta-Llama-3-8B-Instruct | 8B | General | Excellent reasoning, strong benchmarks, good instruction following | Not medical-specific | Runs with 4-bit quantization | Best overall |
| mistralai/Mistral-7B-Instruct-v0.2 | 7B | General | Fast inference, strong reasoning, widely used in RAG systems | Slightly weaker knowledge depth | Runs easily on T4 | Best for speed |
| epfl-llm/meditron-7b | 7B | Medical | Trained on PubMed and clinical texts | Weaker instruction following | Runs on T4 | Good domain baseline |
| BioMistral-7B | 7B | Medical | Biomedical pretraining improves performance over MediTron on some tasks | Some hallucination issues | Runs on T4 | Good medical candidate |
| OpenBioLLM-8B | 8B | Medical | Llama-3 based medical tuning | Less widely tested | Runs on T4 with quantization | Promising experimental |

In this step, the task requires the model to act as a medical assistant answering natural language questions. Based on Strengths and Weeknesses in the table, **meta-llama/Meta-Llama-3-8B-Instruct** is selected for response generation for its strong ability to generate:

* Structured explanations

* Follow prompts correctly

* Produce step-by-step reasoning


#### **Downloading the model from Hugging Face**

In [ ]:
model_name_or_path = "bartowski/Meta-Llama-3-8B-Instruct-GGUF"
model_basename = "Meta-Llama-3-8B-Instruct-Q4_K_M.gguf" # the model is in gguf format

In [ ]:
from huggingface_hub import login
login()

In [ ]:
bartowski_model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

Meta-Llama-3-8B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

#### **Model Configuration**
* Max context of 5000 tokens (Max allowed 8192 Tokens)
* Llama 3 8B has 32 transformer layers so 38 layers means every layer runs on GPU
* Process 512 tokens batch in parallel

In [ ]:
llm = Llama(
    model_path=bartowski_model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Function to generate response

I have set model parameters for **deterministic** response.

* temperature=0 - always pick highest probability token
* top_p=0.95 - probability mass cutoff
* top_k=10 means: only consider top 10 tokens
* Max output tokens 256, Response gets truncated beyond that

In [ ]:
def llm_response(query,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

#### **Declare Query Constants**

In [ ]:
from typing import Final
Query1: Final[str] = "What is the protocol for managing sepsis in a critical care unit?"
Query2: Final[str] = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
Query3: Final[str] = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
Query4: Final[str] = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
Query5: Final[str] = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

### Loading Vector Embedding Collection in Chromadb

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import chromadb

chroma_path = "/content/drive/MyDrive/Colab Notebooks/Medical Assistant/chroma_db"

def get_collection(chunk_size=512, overlap=32):
    collection_name = f"medical_assistant-{chunk_size}.{overlap}"
    client     = chromadb.PersistentClient(path=chroma_path)
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}   # cosine similarity
    )

    # ── Only embed and add if collection is empty ─────────────
    if collection.count() > 0:
        print(f"Loaded existing collection with {collection.count()} chunks")

    return collection

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

try:
    embedding_model
except NameError:
    embedding_model = SentenceTransformer("all-mpnet-base-v2")

def generate_embeddings(all_chunks):
  chunk_texts = [chunk["text"] for chunk in all_chunks]
  embeddings = embedding_model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)
  return embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:

def chroma_retrieve(query, collection, k=20):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = k,
        include          = ["documents", "metadatas", "distances"]
    )

    return [
        {
            "text":     doc,
            "metadata": meta,
            "score":    1 - dist        # convert distance to similarity score
        }
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
    ]

In [ ]:
collection = get_collection()

Loaded existing collection with 8125 chunks


## Re Ranking

Now we have the probable relevant top values, we need to reorder the top retrieved documents based on deeper semantic relevance. We will use a **cross-encoder** to reorder search results to help improve clinical precision.
We will pass the cross-encoder re-ranking model, query + document pairs together as input and outputs a single relevance score.

Re-ranking improves precision context relevance downstream generation quality reduction of hallucinations


In [ ]:
from sentence_transformers import CrossEncoder
try:
  reranker
  print("reranker already loaded")
except NameError:
  reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")

config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
def rerank(query, relevant_document_chunks):
    pairs = [[query, chunk["text"]] for chunk in relevant_document_chunks]
    scores = reranker.predict(pairs)
    ranked_idx = np.argsort(scores)[::-1]
    return [relevant_document_chunks[i] for i in ranked_idx[:5]]

## **Generation**

### **Design Decision: Choosing Generation Model (Decoder)**

We will choose meta-llama/Meta-Llama-3-8B-Instruct which is same as the model we initially started with ("bartowski/Meta-Llama-3-8B-Instruct-GGUF") just in a different format.

The model is selected based on:

* Very strong reasoning ability

* Excellent instruction following

* Handles complex explanations well

Model is suppose to perform well without medical specialization as
medical knowledge exists in general training data, reasoning quality compensates for lack of specialization
and it works very well with RAG and prompt engineering.

### **Design Decision: Context Window**

Llama allows context window of 8192 tokens. We need max 5000 tokens for k=3 to 5 for chunk size of 512

(num_chunks × chunk_size) + system_prompt + question + max_new_tokens < context_window

In [ ]:
llm = Llama(
    model_path=bartowski_model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


# Question Answering using RAG

### System and User Prompt Template

In [ ]:
qna_system_message = """<|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>
"""

In [ ]:
qna_user_message_template = """<|start_header_id|>user<|end_header_id|>
<context>
#context
</context>
<question>
#question
</question>
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

### Response Function

In [ ]:
def create_prompt(relevant_document_chunks, system_prompt, user_prompt_template, user_input):

    context_for_query = ". ".join([chunk["text"] for chunk in relevant_document_chunks])

    user_message = user_prompt_template.replace('#context', context_for_query)
    user_message = user_message.replace('#question', user_input)

    prompt = system_prompt + '\n' + user_message

    return prompt

In [ ]:
def generate_rag_response(prompt,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k,
                  stop        = ["<|eot_id|>",
                           "<|start_header_id|>",  "\nassistant", "\nuser"]
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query1, collection,3)
pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query1, relevant_document_chunks)

── Chunk 1 ──────────────────────────────────────────
Page   : 2457
Score  : 0.6607
Text   :
Parenteral antibiotics should be given after specimens of blood, body fluids,
and wound sites have been taken for Gram stain and culture. Very prompt empiric
therapy, started immediately after suspecting sepsis, is essential and may be
lifesaving. Antibiotic selection requires an educated guess based on the
suspected source, clinical setting, knowledge or suspicion of causative
organisms and of sensitivity patterns common to that specific inpatient unit,
and previous culture results. One regimen for septic shock of unknown cause is
gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation
cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day or, if
Pseudomonas is suspected, ceftazidime 2 g IV q 8 h). Alternatively, ceftazidime
plus a fluoroquinolone (eg, ciprofloxacin) may be used. Monotherapy with maximal
therapeutic doses of ceftazidime (2 g IV q 8 h) or imipenem (1 

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
print(prompt)
print(generate_rag_response(prompt));

<|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
Chapter 227. Sepsis and Septic Shock Sepsis, severe sepsis, and septic shock are inflammatory states resulting from the systemic response to bacterial infection. In severe sepsis and septic shock, there is critical reduction in tissue perfusion. Common causes include gram-negative organisms, staphylococci, and meningococci. Symptoms often begin with shaking chills and include fever, hypotension, oliguria, and confusion. Acute fail

## **Fine-tuning**

Fine tuning the RAG Architecture, involve picking the right model, **optimal chunking**, **optimal retirval** and **generation**. Below are the 3 different combination that we will try with **Deterministic** and **Conservative Hyperparameter** with **Constrainted Prompting** these we will try to achieve optimal results
| Experiment | Chunk Size | Overlap | TopK | Embedding Model      | LLM Model|
|------------|------------|---------|------|----------------------|------------------------|
| 1          | 256        | 32      | 3    | all-mpnet-base-v2    | Llama-3-8B-Instruct    |
| 2          | 512        | 64      | 5    | all-mpnet-base-v2    | Llama-3-8B-Instruct    |
| 3          | 768        | 64      | 5    | all-mpnet-base-v2    | Llama-3-8B-Instruct    |

We will perform above three experiments with **Deterministic** and **Conservative** LLM configuraiton. Since this is Medical Assistant to provide tools to support quick decision-making and enhance efficiency, we will experiment with towards Deterministic settings.


| Strategy | temperature | top_p | top_k | max_tokens | Use Case |
|---|---|---|---|---|---|
| Deterministic | 0 | 0.95 | 10 | 256 | Factual, consistent answers |
| Conservative | 0.1 | 0.9 | 20 | 256 | Slight variation, still safe |


## **Configuration 1**


This is base case for **Deterministic behavior**. Max output token is 256 which can be a challenge as it potentially can truncate response.

In [ ]:
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt) # k=3,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
print(response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In a critical care unit, the protocol for managing sepsis involves a multi-disciplinary approach, including aggressive fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues, supportive care, and intensive control of blood glucose.

**Treatment Protocol**

1. **Initial Assessment**: Patients with suspected sepsis should be evaluated promptly, and specimens of blood, body fluids, and wound sites should be taken for Gram stain and culture.
2. **Empiric Antibiotic Therapy**: Parenteral antibiotics should be given immediately after suspecting sepsis, based on the suspected source, clinical setting, knowledge or suspicion of causative organisms, and previous culture results.
3. **Fluid Resuscitation**: Aggressive fluid resuscitation is essential to restore tissue perfusion. A fluid challenge should be done initially, followed by further fluid therapy

## **Configuration 2**

* Notice the temperature=0.1,
* top_p=0.9.
* Chunck and overlap stays same.
* Reranking model and configuration stays same.

In [ ]:
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt, max_tokens=256,temperature=0.1,top_p=0.9,top_k=20)
print(response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In a critical care unit, the protocol for managing sepsis involves a multi-disciplinary approach, including aggressive fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues, supportive care, and intensive control of blood glucose.

**Treatment Protocol**

1. **Initial Management**: Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been taken for Gram stain and culture.
2. **Empiric Therapy**: Prompt empiric therapy is essential and may be lifesaving. Antibiotic selection requires an educated guess based on the suspected source, clinical setting, knowledge or suspicion of causative organisms, and previous culture results.
3. **Antibiotic Regimen**: For septic shock of unknown cause, a regimen consisting of gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxi

## **Configuration 3**

* Notice the shift to Deterministic setting temperature=0,top_p=0.95.
* Chunks and Overlap increases to 768 and 64

In [ ]:
collection = get_collection(768, 64);
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt, max_tokens=768,temperature=0,top_p=0.95,top_k=10)
print(response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In a critical care unit, the goal of treatment is to prevent organ dysfunction and death by addressing the underlying infection, restoring perfusion, and managing inflammation.

* Sepsis is defined as a systemic inflammatory response syndrome (SIRS) caused by an infectious agent or its toxins.
* The diagnosis of sepsis is based on clinical criteria, including:
	+ Two or more SIRS criteria (e.g., fever, tachycardia, tachypnea)
	+ Evidence of infection (e.g., positive blood culture, radiographic evidence of pneumonia)

**Treatment Protocol**

The management of sepsis in a critical care unit involves a multidisciplinary approach, including:

* **Initial Resuscitation**
	+ Administer 30 mL/kg of crystalloid solution over 1 hour to restore perfusion and prevent organ dysfunction
	+ Monitor for signs of fluid overload (e.g., increased blood pressure, decreased urine output)
* **A

## **Configuration 4**

* Notice the shift to Conservative setting temperature=0.1,top_p=0.9
* Chunks and Overlap stays same to 768 and 64
* Top K stays at 10
* Top N (After Reranking) stays at 3
* Lower number compensate for higher token size of 512

In [ ]:
## Using same collection for chunks_size=768, overlap=64
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt,max_tokens=512,temperature=0.1,top_p=0.9,top_k=20)
print(response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that occurs when an overwhelming inflammatory response to infection leads to organ dysfunction and potentially death. Early recognition and management of sepsis are crucial to improve outcomes.

The Surviving Sepsis Campaign (SSC) guidelines recommend a standardized approach to managing sepsis in critical care units. The goal is to identify and treat sepsis promptly, aiming for early resuscitation and organ support.

**Treatment Protocol**

1. **Initial Assessment**
	* Perform a thorough physical examination and review laboratory results.
	* Identify the source of infection (if possible).
2. **Resuscitation**
	* Administer 30 mL/kg of crystalloid solution within the first hour to achieve a central venous pressure (CVP) of 8-12 mmHg or a pulmonary artery occlusion pressure (PAOP) of 18-22 mmHg.
	* Monitor and adjust fluid administration based on hemodynamic response.
3. **Antimicrobial Therapy**
	* Administer broad-spectru

## **Configuration 5**

* Chunk size: 512
* Overlap: 64
* Embedding model: all-mpnet-base-v2
* Vector DB: ChromaDB
* LLM: Llama-3-8B-Instruct (GGUF)
* Temperature: 0
* Max tokens: 512
* Prompt: grounded medical system prompt

In [ ]:
collection = get_collection(512,64)
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks1 = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks1,qna_system_message,qna_user_message_template, Query1)
Query1_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=3)
print(Query1_response)

Loaded existing collection with 8200 chunks


Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In a critical care unit, the protocol for managing sepsis involves a multi-faceted approach that includes:

* Aggressive fluid resuscitation with 0.9% normal saline to maintain adequate blood pressure and perfusion
* Broad-spectrum antibiotics administered empirically based on suspected source of infection and clinical setting
* Drainage of abscesses and excision of necrotic tissue to eliminate septic foci
* Normalization of blood glucose levels through continuous IV insulin infusion
* Replacement-dose corticosteroids to support adrenal function

**Treatment Protocol**

1. **Initial Assessment**: Rapidly assess the patient's condition, including vital signs, laboratory results, and clinical presentation.
2. **Fluid Resuscitation**: Administer 0.9% normal saline at a rate of 500-1000 mL/h to maintain adequate blood pressure and perfusion.
3. **Antibiotic Therapy**: Initiate 

| Configuration | Name | Chunk Size | Overlap | Temperature | top_p | top_k | max_tokens | Response Quality | Issues |
|---|---|---|---|---|---|---|---|---|---|
| 1 | Small Chunks Deterministic | 256 | 32 | 0 | 0.95 | 10 | 256 | Good structure, covers key protocol steps | Cut off mid-answer, small chunks miss context |
| 2 | Small Chunks Conservative | 256 | 32 | 0.1 | 0.90 | 20 | 256 | More detailed, adds corticosteroids and vasopressors | Cut off mid-sentence |
| 3 | Large Chunks Deterministic | 768 | 64 | 0 | 0.95 | 10 | 256 | Adds activated protein C therapy | Cut off mid-sentence, outdated treatment (withdrawn 2011) |
| 4 | Large Chunks Conservative | 768 | 64 | 0.1 | 0.90 | 20 | 512 | Most complete — cites chapters, blood glucose targets | Stop token bug causes self-evaluation after answer |
| 5 | Medium Chunks Deterministic | 512 | 64 | 0 | 0.95 | 5 | 512 | Balanced chunk size and deterministic generation optimized for accuracy | Best trade-off between context quality, retrieval precision, and response length. |


In [ ]:
# Query2 Response
relevant_document_chunks = chroma_retrieve(Query2, collection)
reranked_chunks2 = rerank(Query2, relevant_document_chunks)
prompt = create_prompt(reranked_chunks2,qna_system_message,qna_user_message_template, Query2)
Query2_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query2_response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Appendicitis is characterized by acute inflammation of the vermiform appendix, typically resulting in abdominal pain, anorexia, and abdominal tenderness. The classic symptoms of acute appendicitis are epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Additional signs include right lower quadrant direct and rebound tenderness located at McBurney's point, Rovsing sign, psoas sign, and obturator sign.

**Treatment Protocol**

Appendicitis is typically treated with surgical removal of the appendix, either through open or laparoscopic appendectomy. The surgeon can usually remove the appendix even if perforated. IV fluids and antibiotics are administered preoperatively to help manage symptoms and prevent complications.

For nonperforated appendicitis, no further antibiotics are required after surgery. If the appendix is perforated, antibiotics should be continued until t

In [ ]:
# Query3 Response
relevant_document_chunks = chroma_retrieve(Query3, collection)
reranked_chunks3 = rerank(Query3, relevant_document_chunks)
prompt = create_prompt(reranked_chunks3,qna_system_message,qna_user_message_template, Query3)
Query3_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query3_response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder characterized by the sudden onset of patchy hair loss on the scalp or other hairy areas. It can occur at any age but is most common in children and young adults.

The possible causes behind alopecia areata include:

* Genetic predisposition
* Environmental triggers (e.g., stress, infection)
* Hormonal changes
* Autoimmune disorders

**Treatment Protocol**

Effective treatments for alopecia areata include:

1. **Topical corticosteroids**: Triamcinolone acetonide suspension can be injected intradermally or potent topical corticosteroids like betamethasone 0.05% bid can be used.
2. **Minoxidil**: Topical minoxidil 1 mL bid applied to the scalp is most effective for vertex alopecia in male-pattern or female-pattern hair loss.
3. **Anthralin**: Topical anthralin (0.5 to 1% for 10 to 20 min daily, then washed off) can be used.
4. **Immunotherapy**: Induction of allergic contact dermati

In [ ]:
# Query4 Response
relevant_document_chunks = chroma_retrieve(Query4, collection)
reranked_chunks4 = rerank(Query4, relevant_document_chunks)
prompt = create_prompt(reranked_chunks4,qna_system_message,qna_user_message_template, Query4)
Query4_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query4_response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Traumatic brain injury (TBI) is a physical injury to brain tissue that temporarily or permanently impairs brain function. The initial treatment consists of ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.

**Treatment Protocol**

* Ensure a reliable airway
* Maintain adequate ventilation, oxygenation, and blood pressure
* Monitor for and manage increased intracranial pressure (ICP)
* Consider surgery in patients with more severe injuries to:
	+ Place monitors to track ICP
	+ Decompress the brain if ICP is increased
	+ Remove intracranial hematomas
* Maintain adequate brain perfusion and oxygenation
* Prevent complications of altered sensorium

**Cite**

Chapter 324. Traumatic Brain Injury, The Merck Ma

In [ ]:
# Query5 Response
relevant_document_chunks = chroma_retrieve(Query5, collection)
reranked_chunks5 = rerank(Query5, relevant_document_chunks)
prompt = create_prompt(reranked_chunks5,qna_system_message,qna_user_message_template, Query5)
Query5_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query5_response)

Llama.generate: prefix-match hit


Clinical Explanation:

A fracture of the leg can occur due to various reasons such as trauma, overuse, or osteoporosis. The symptoms may include pain, swelling, bruising, and limited mobility in the affected limb. It is essential to seek medical attention immediately if you suspect a fracture.

Treatment Protocol:

1. Immobilization: Apply a splint or cast to immobilize the leg and reduce pain.
2. Pain management: Use analgesics such as acetaminophen or NSAIDs to manage pain.
3. Rest: Avoid putting weight on the affected limb and rest it as much as possible.
4. Elevation: Elevate the affected limb above the level of the heart to reduce swelling.
5. Ice application: Apply ice packs to the affected area for 15-20 minutes, several times a day, to reduce pain and inflammation.
6. Compression: Use an elastic bandage or compression wrap to compress the affected area and reduce swelling.
7. Rehabilitation: Gradually increase mobility and strength exercises under the guidance of a healthcare p

# Output Evaluation

We will use the judge model to evaluates:

* Faithfulness – Is the answer supported by retrieved context?
* Relevance – Did it answer the question?
* Medical correctness – Is the treatment aligned with the manual?
* Completeness – Did it miss critical protocol steps?
* Hallucination detection – Did it add unsupported claims?

The right judge model requires strong reasoning, strong instruction following, stable output, not necessarily huge parameter size. It can't be same model as generation model as that will introduce bias. I will use meta-llama/Meta-Llama-3-8B-Instruct because of its stability, performance, strong reasoning and it can perform with Colab T4.

In [ ]:
judge_model_name_or_path="TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
judge_model_basename="mistral-7b-instruct-v0.2.Q4_K_M.gguf"

In [ ]:
judgellm_model_path = hf_hub_download(
    repo_id=judge_model_name_or_path,
    filename=judge_model_basename
)

mistral-7b-instruct-v0.2.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

In [ ]:
#uncomment the below snippet of code if the runtime is connected to GPU.
judge_llm = Llama(
    model_path=judgellm_model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


## Groundedness

Below groundedness rater prompt uses xml tags making them less ambigious. The exmaples (Few-Shot prompt engineering) are provided as reference to model. Model has been told to produce json output in very structured expected format.

We are going to use our knowledge base created from Merck manual to generate an example for prompt.

In [ ]:
groundedness_rater_system_message = """
You are a strict medical QA evaluator.

You will be given a <question>, <context>, and an AI-generated <answer>.

Your task: Judge whether the <answer> is derived from and supported by the <context>.

Scoring rubric:
- Score 1: Answer contradicts the context or introduces facts not present in context
- Score 2: Answer is loosely related to context but makes unsupported claims
- Score 3: Answer uses context but includes some details not found in context
- Score 4: Answer is mostly grounded in context with minor gaps
- Score 5: Every claim in the answer is directly supported by the context

## Examples

<question>What causes Type 2 diabetes mellitus in adolescents?</question>
<context>Type 2 diabetes mellitus is occurring with increasing frequency due to obesity in adolescents</context>
<answer>Type 2 diabetes mellitus is occurring with increasing frequency due to obesity in adolescents</answer>
{"score": 5, "reason": "Claim directly stated in context.", "unsupported_claims": "none"}

---
<question>What causes Type 2 diabetes?</question>
<context>Type 2 diabetes mellitus is occurring with increasing frequency due to obesity in adolescents</context>
<answer>Type 2 diabetes is caused by sedentary lifestyle.</answer>
{"score": 1, "reason": "Sedentary lifestyle not mentioned in context.", "unsupported_claims": "sedentary lifestyle"}

---
IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
IMPORTANT: Respond ONLY in this exact JSON format, no other text:
{
  "score": <integer 1-5>,
  "reason": "<one sentence explanation>",
  "unsupported_claims": "<list any claims not found in context, or 'none'>"
}
"""

In [ ]:
relevance_rater_system_message = """
You are a strict medical QA evaluator.

You will be given a <question>, <context>, and an AI-generated <answer>.

Your task: Judge whether the <answer> directly and completely addresses the <question>.

Scoring rubric:
- Score 1: Answer is completely off-topic or does not address the question
- Score 2: Answer addresses the topic but misses the core ask of the question
- Score 3: Answer partially addresses the question but omits key aspects
- Score 4: Answer addresses the question well with minor omissions
- Score 5: Answer directly and completely addresses all aspects of the question

## Examples

<question>What are the symptoms of hypertension?</question>
<context>Hypertension often has no symptoms but can cause headaches and vision problems.</context>
<answer>Hypertension can cause headaches and vision problems, and is often asymptomatic.</answer>
{"score": 5, "reason": "Directly answers all aspects of the question.", "missing_aspects": "none"}

---
<question>What are the symptoms of hypertension?</question>
<context>Hypertension often has no symptoms but can cause headaches and vision problems.</context>
<answer>Hypertension is a serious cardiovascular condition affecting millions.</answer>
{"score": 1, "reason": "Does not address symptoms at all.", "missing_aspects": "all symptoms"}

---
<question>What are the symptoms of hypertension?</question>
<context>Hypertension often has no symptoms but can cause headaches and vision problems.</context>
<answer>Hypertension can cause headaches.</answer>
{"score": 3, "reason": "Mentions headaches but omits vision problems and asymptomatic nature.", "missing_aspects": "vision problems, asymptomatic cases"}

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
IMPORTANT: Respond ONLY in this exact JSON format, no other text:
{
  "score": <integer 1-5>,
  "reason": "<one sentence explanation>",
  "missing_aspects": "<what key aspects of the question were not addressed, or 'none'>"
}
"""

In [ ]:
faithfulness_rater_system_message = """
You are a strict medical QA evaluator.

You will be given a <question>, <context>, and an AI-generated <answer>.

Your task: Detect whether the <answer> contains hallucinated facts —
claims that are NEITHER supported by the <context> NOR established medical knowledge.

## Examples

<question>What is the treatment for Type 2 diabetes?</question>
<context>Type 2 diabetes is managed with metformin, lifestyle changes, and blood sugar monitoring.</context>
<answer>The primary treatment for type 2 DM is oral antihyperglycemic drugs</answer>
{"is_faithful": true, "hallucinated_claims": "none", "severity": "none"}

---
<question>What is the treatment for Type 2 diabetes?</question>
<context>Type 2 diabetes is managed with metformin, lifestyle changes, and blood sugar monitoring.</context>
<answer>Type 2 diabetes is cured by insulin injections taken three times daily.</answer>
{"is_faithful": false, "hallucinated_claims": "insulin injections three times daily, claimed as cure", "severity": "major"}

---
<question>What is the treatment for Type 2 diabetes?</question>
<context>Type 2 diabetes is managed with metformin, lifestyle changes, and blood sugar monitoring.</context>
<answer>Type 2 diabetes is treated with metformin. Patients should also avoid sugar completely.</answer>
{"is_faithful": false, "hallucinated_claims": "avoid sugar completely", "severity": "minor"}

CRITICAL INSTRUCTIONS:
- Output ONLY one JSON object
- Do NOT change your answer after producing JSON
- Do NOT apologize or revise
- Stop immediately after the closing brace }

...scoring rubric and examples...

IMPORTANT: Respond ONLY in this exact JSON format:
{
  "is_faithful": <true or false>,
  "hallucinated_claims": "<claims or none>",
  "severity": "<none | minor | major>"
}
"""

In [ ]:
# Single template used across all 3 raters
judge_user_message_template = """
<question>{question}</question>
<context>{context}</context>
<answer>{answer}</answer>
"""

In [1]:
def build_mistral_prompt(system_message, user_message):
    """
    Mistral-Instruct-v0.2 expects [INST] ... [/INST] format.
    System content is folded into the [INST] block — Mistral v0.2
    does not have a dedicated system role token.
    """
    return f"<s>[INST] {system_message.strip()}\n\n{user_message.strip()} [/INST]"

In [3]:
def evaluate_with_judge(judge_llm, question, answer,
                        reranked_chunks, max_tokens=150):
    raters = {
        "groundedness": groundedness_rater_system_message,
        "relevance":    relevance_rater_system_message,
        "faithfulness": faithfulness_rater_system_message,
    }

    context_for_query = ". ".join([chunk["text"] for chunk in reranked_chunks])

    judge_user_message = judge_user_message_template.format(
        question = question,
        context  = context_for_query,
        answer   = answer
    )

    results = {}
    for metric, system_msg in raters.items():
        prompt = build_mistral_prompt(system_msg, judge_user_message)
        raw = judge_llm(
            prompt      = prompt,
            max_tokens  = max_tokens,
            temperature = 0,
            top_p       = 0.95,
            top_k       = 10,
            stop        = ["</s>", "[INST]"],   # Mistral stops, not Llama-3 stops
              )["choices"][0]["text"].strip()

        # Extract first JSON block only
        try:
            json_start = raw.find("{")
            json_end   = raw.find("}") + 1
            if json_start == -1 or json_end == 0:
                raise ValueError("No JSON found")
            clean = raw[json_start:json_end]
            results[metric] = json.loads(clean)
        except (json.JSONDecodeError, ValueError):
            results[metric] = {"error": "parse_failed", "raw": raw}

    return results

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
evaluate_with_judge(judge_llm,Query1, Query1_response, reranked_chunks1)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 5,
  'reason': 'Answer directly derived from context.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly addresses the question by outlining the steps involved in managing sepsis in a critical care unit.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': True,
  'hallucinated_claims': 'none',
  'severity': 'none'}}

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
evaluate_with_judge(judge_llm,Query2, Query2_response, reranked_chunks2)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 5,
  'reason': 'Answer is directly supported by the context.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly answers all aspects of the question.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': False,
  'hallucinated_claims': 'IV fluids and antibiotics as treatment for non-perforated appendicitis, antibiotics not curative in cases where surgery is impossible',
  'severity': 'minor'}}

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
evaluate_with_judge(judge_llm,Query3, Query3_response, reranked_chunks3)

Loaded existing collection with 8200 chunks


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 3,
  'reason': 'The context does not directly discuss alopecia areata or its causes and treatments. However, the answer mentions some relevant information about this condition.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly addresses the question by providing information on treatments for sudden patchy hair loss and possible causes.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': True,
  'hallucinated_claims': 'none',
  'severity': 'none'}}

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
evaluate_with_judge(judge_llm,Query4, Query4_response, reranked_chunks4)

Loaded existing collection with 8200 chunks


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 5,
  'reason': 'The answer directly repeats the information from the context about the recommended treatments for traumatic brain injury.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly and completely addresses the question by detailing the recommended treatments for a person with a brain injury.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': True,
  'hallucinated_claims': 'none',
  'severity': 'none'}}

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
evaluate_with_judge(judge_llm,Query5, Query5_response, reranked_chunks5)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 2,
  'reason': 'Answer is loosely related to context but makes unsupported claims',
  'unsupported_claims': 'The answer does not specifically address a fractured leg during a hiking trip, and some treatment protocols mentioned are not directly applicable to this scenario.'},
 'relevance': {'score': 2,
  'reason': 'The answer partially addresses the question by providing general information about treating various types of injuries and fractures, but does not specifically address a fractured leg during a hiking trip.',
  'missing_aspects': 'Specific treatment protocol for a fractured leg during a hiking trip'},
 'faithfulness': {'is_faithful': False,
  'hallucinated_claims': 'none',
  'severity': 'minor'}}

| Query | Question | Groundedness | Relevance | Faithfulness | Overall |
|---|---|---|---|---|---|
| Q1 | Managing sepsis in critical care unit | 5/5 | 5/5 | False (minor) | Minor faithfulness concern |
| Q2 | Symptoms & treatment for appendicitis | 5/5 | 5/5 | False (minor) | Minor hallucination detected |
| Q3 | Sudden patchy hair loss treatments | 3/5 | 5/5 | True | Retrieval mismatch |
| Q4 | Treatments for traumatic brain injury | 5/5 | 5/5 | True | Best performing |
| Q5 | Precautions for fractured leg hiking | 2/5 | 2/5 | False (minor) | Retrieval failure |

## Observation and Insights

1. Retrieval-Augmented Generation (RAG) significantly improves medical answer accuracy

Experiments show that a standalone LLM without context produces generic responses, occasional hallucinations, missing medical references

After implementing RAG with medical documents (Merck Manuals) responses become more precise answers include domain-specific terminology and hallucination risk is reduced.

**Insight**: Healthcare AI systems must be grounded in trusted medical sources rather than relying solely on pretrained model knowledge.

**Impact**: Higher reliability of clinical information, Reduced risk of misinformation, and better trust from medical professionals

2. Document chunking and embeddings directly affect information retrieval quality

Experiments with chunking and embedding configurations demonstrates that semantic chunking + embedding models allow the system to retrieve relevant medical passages before generating an answer.

Poor chunking can lead to incomplete context, irrelevant retrieval, and weaker answers

**Insight**: The quality of the retrieval layer is as important as the LLM itself.

**Impact**: To build a medical AI assistants we must invest in effort high-quality document preprocessing optimized chunking strategies and domain-specific embedding models

3. Re-Ranking improves context relevance for complex medical queries

By introducing cross-encoder re-ranking, the system filters retrieved chunks to ensure the most relevant medical passages are used. Without re-ranking vector search may retrieve partially relevant documents

With re-ranking, higher contextual precision and more accurate answers can be achieved

**Insight**: Multi-stage retrieval pipelines significantly improve AI reliability for complex knowledge domains like healthcare.

**Impact**: This architecture reduces the chance that incorrect or irrelevant passages influence the final answer.

4. LLM Prompt Engineering Improves Safety and Response Quality

**prompt engineering** helps ensure that the system uses only retrieved context,that hallucinations are minimized and answers follow structured medical explanations

**Insight**: Prompt engineering is a critical control layer for medical AI safety.

**Impact**: Organizations deploying AI assistants should implement strict prompting frameworks that enforce, context grounding, structured answers and source citations

<font size=6 color='blue'>Power Ahead</font>
___